# OSML Jupyter Extension - Notebook Viewer API Example

This notebook demonstrates the programmatic `viewer` object exposed at `aws.osml.jupyter`. It lets you drive the OSML image viewer from an attached notebook: read the live view state (current image, settled view bounds, last click + clicked feature), reach the underlying toolkit objects for the current image (`reader` / `sensor_model` / `chip_factory`), query imagery, and push results back as rendered layers or viewport moves.

## Prerequisites (viewer-first topology)

1. **Open an image first** using the OSML Jupyter Extension context menu (right-click a raster in the file browser → "OversightML: Open").
2. **Attach this notebook to the viewer's kernel** — select the same kernel the image viewer launched. The `viewer` object binds to that kernel's live state.
3. Run the cells below. Reading or acting on `viewer` before an image is open raises `ViewerError`.

> **Note:** The supported topology is *viewer-first* — open an image, then attach a notebook. Starting a notebook first and attaching a viewer to it is out of scope.

In [ ]:
from aws.osml.jupyter import viewer, ViewerError

# The current image the viewer has open (dataset path, or None).
print(viewer.current_image)

## 1. Read the live view state

As you navigate in the viewer, the frontend streams the settled view bounds and the last click back to the kernel. `viewer.view_bounds` and `viewer.last_click` reflect that state. They return `None` until the view has settled / a click has landed.

State reads are returned in the toolkit's own types — `view_bounds.image` is a `shapely` box, `last_click.image` is an `ImageCoordinate` — for consistency with what you already import.

In [ ]:
# Navigate the viewer, then re-run this cell after the view settles (~300 ms).
bounds = viewer.view_bounds
if bounds is None:
    print("No settled view yet — pan or zoom the viewer, then re-run this cell.")
else:
    print("image-space rect:", bounds.image)  # shapely box (minx, miny, maxx, maxy)
    print("zoom:", bounds.zoom)

### Lazy world enrichment

The image → world transform (sensor model) is computed **lazily, only when you read `.world`**, and is memoized per bounds / per click. `view_bounds.world` is **four corner `GeodeticWorldCoordinate`s** in image-corner order (top-left, top-right, bottom-right, bottom-left) — a general (possibly rotated/skewed) quadrilateral, **not** an axis-aligned lat/lon bbox. `view_bounds.world_polygon` wraps those same corners as a GeoJSON `Polygon` (in degrees) for convenience.

> **⚠️ `.image` and `.world` do *not* share vertex ordering.** `view_bounds.image` is a **shapely box**, and shapely prints its ring in *its own fixed convention* — starting at `(maxx, miny)` and winding counter-clockwise — regardless of coordinate meaning. `view_bounds.world` / `world_polygon` use an explicit **TL, TR, BR, BL** image-corner order. Both describe the *same* rectangle, but their vertex sequences start at different corners and wind in opposite directions, so **don't compare the two rings vertex-by-vertex**.
>
> For order-independent extent, use **`view_bounds.image.bounds`** → `(minx, miny, maxx, maxy)`. To pair image corners with world corners, build the image corners yourself in the same TL/TR/BR/BL order (see the cell below) rather than reading them off the shapely ring.

In [ ]:
import math

if bounds is not None:
    corners = bounds.world  # 4x GeodeticWorldCoordinate (TL, TR, BR, BL)
    print("world corners (radians):", corners)
    print("world polygon (degrees):", bounds.world_polygon)

    # Order-independent extent — always prefer this over the shapely ring.
    minx, miny, maxx, maxy = bounds.image.bounds
    print("\nimage extent (minx, miny, maxx, maxy):", (minx, miny, maxx, maxy))

    # To pair image corners with world corners, build the image corners in the
    # SAME TL/TR/BR/BL order that .world uses. Do NOT read them off the shapely
    # ring: shapely prints its box starting at (maxx, miny) winding CCW, which
    # is a different vertex sequence than .world — same rectangle, different order.
    image_corners = [(minx, miny), (maxx, miny), (maxx, maxy), (minx, maxy)]
    print("\nimage corner -> world corner (TL, TR, BR, BL):")
    for label, (ix, iy), w in zip(['TL', 'TR', 'BR', 'BL'], image_corners, corners):
        print(
            f"  {label}: image=({ix:.2f}, {iy:.2f}) -> "
            f"lon={math.degrees(w.longitude):.6f}, lat={math.degrees(w.latitude):.6f}"
        )

> **⚠️ Radians caveat.** `GeodeticWorldCoordinate` stores longitude/latitude in **radians**. The viewer's `.world` properties return correctly-constructed coordinates, and `.world_polygon` already emits **degrees** for GeoJSON. But if you call the sensor model directly (see below), the result is in radians — convert it with `math.degrees()`.

In [ ]:
# Click in the viewer (optionally on a feature), then re-run this cell.
click = viewer.last_click
if click is None:
    print("No click yet — click in the viewer, then re-run this cell.")
else:
    print("image coords:", click.image)   # ImageCoordinate
    print("world (lazy):", click.world)    # GeodeticWorldCoordinate (radians)
    print("feature:", click.feature)       # GeoJSON feature, if a feature was hit
    print("layer:", click.layer)           # layer id, if a feature was hit

## 2. Reach the toolkit objects for the current image

`viewer.reader`, `viewer.sensor_model`, and `viewer.chip_factory` hand you the live toolkit objects for the current image — the same `DatasetReader` / `SensorModel` / `ChipFactory` the viewer itself uses. These raise a clear `ViewerError` ("No image loaded" / "Image has no sensor model") rather than returning `None`, so your code fails loudly.

In [ ]:
import math
from aws.osml.photogrammetry import ImageCoordinate

reader = viewer.reader
sensor_model = viewer.sensor_model

# Direct sensor-model transform — the result is in radians, so convert to degrees.
world = sensor_model.image_to_world(ImageCoordinate([100.0, 100.0]))
lon_deg = math.degrees(world.longitude)
lat_deg = math.degrees(world.latitude)
print(f"pixel (100, 100) -> lon={lon_deg:.6f}, lat={lat_deg:.6f} (degrees)")

## 3. Query the current image

`viewer.metadata()` and `viewer.statistics()` return the current image's metadata and statistics. `viewer.features_in(bbox=None, layer=None)` returns the features intersecting a bbox (image-space shapely geometry or `(minx, miny, maxx, maxy)` tuple); it defaults to the current view bounds and searches every overlay loaded for the current image unless you restrict it to a single `layer`.

In [ ]:
print("metadata keys:", list(viewer.metadata().keys()))
print("statistics:", viewer.statistics())

# Features intersecting the current view (defaults to viewer.view_bounds).
if viewer.view_bounds is not None:
    print("features in view:", len(viewer.features_in()))

## 4. Push results back as a rendered layer

`viewer.add_layer(features, name)` projects and indexes features on the kernel, then pushes an `ADD_LAYER` render command to the viewer — the features render immediately over the existing tile path. It accepts a GeoJSON `FeatureCollection` dict, a `list` of features, or any object exposing `__geo_interface__` (a GeoDataFrame is duck-typed — no hard geopandas dependency).

`name` is **required**; re-adding under the same name **replaces** the existing layer, so re-running a cell is idempotent. Feature coordinates use image pixel space (top-left origin, x→, y↓): use `imageBBox` for axis-aligned rectangles and `imageGeometry` for any other shape.

The first cell below uses **`imageBBox`** for bounding boxes — it takes a hardcoded **COCO**-style detection list (`bbox` as `[x_min, y_min, width, height]` in pixels) and converts each box to an `imageBBox`, the small adapter you'd write once for whatever your model emits. The second cell uses **`imageGeometry`** for non-rectangular shapes (lines and polygons).

In [ ]:
import geojson

# A hardcoded COCO-style detection result, as a model might return it. COCO
# `bbox` is [x_min, y_min, width, height] in image pixels.
coco_detections = [
    {"bbox": [1000, 750, 50, 50], "category": "vehicle", "score": 0.95},
    {"bbox": [2000, 1500, 75, 75], "category": "building", "score": 0.87},
]

# Convert each detection to a viewer feature. imageBBox is
# [x_min, y_min, x_max, y_max] in pixels (top-left origin); other fields ride
# along in properties and show up in the property inspector on click.
features = []
for det in coco_detections:
    x, y, w, h = det["bbox"]
    features.append(geojson.Feature(
        geometry=None,
        properties={
            "imageBBox": [x, y, x + w, y + h],
            "object_class": det["category"],
            "confidence": det["score"],
        },
    ))

viewer.add_layer(geojson.FeatureCollection(features=features), name="my_detections")
print(f"Rendered {len(features)} detections as layer 'my_detections'.")

### Non-rectangular features with `imageGeometry`

`imageBBox` only describes axis-aligned rectangles. For anything else — a road as a `LineString`, a region or footprint as a `Polygon`, a point marker as a `Point` — set `imageGeometry` to a GeoJSON geometry whose coordinates are `[x, y]` pixel pairs (same top-left origin). `add_layer` accepts a mix of geometry types in one collection, so a single named layer can carry lines and polygons together.

In [ ]:
import geojson

# imageGeometry takes any GeoJSON geometry in [x, y] pixel coordinates. Here a
# LineString (e.g. a road or path) and a Polygon (e.g. a region or footprint)
# are published together in one layer. A Point marker works the same way.
annotations = geojson.FeatureCollection(features=[
    geojson.Feature(
        geometry=None,
        properties={
            "imageGeometry": {
                "type": "LineString",
                "coordinates": [[100, 200], [500, 200], [500, 800], [900, 800]]
            },
            "name": "Main Street",
            "kind": "road"
        },
    ),
    geojson.Feature(
        geometry=None,
        properties={
            "imageGeometry": {
                "type": "Polygon",
                # A ring must repeat its first coordinate as its last to close.
                "coordinates": [[[500, 500], [1500, 500], [1500, 1200], [500, 1200], [500, 500]]]
            },
            "name": "Urban Area",
            "kind": "region"
        },
    ),
])

viewer.add_layer(annotations, name="my_annotations")
print(f"Rendered {len(annotations['features'])} annotations as layer 'my_annotations'.")

## 5. Move the viewport

`viewer.goto(...)` moves the viewport to a point. Pass either image-space `x`/`y` or geographic `lon`/`lat` (degrees) — geographic coordinates are converted to image space kernel-side via the current sensor model. `zoom` is optional; when omitted the viewer preserves the current zoom. `viewer.set_view(bounds, zoom=None)` centers the viewport on an image-space rectangle.

In [ ]:
# Move to an image pixel with an explicit zoom.
viewer.goto(x=1024, y=768, zoom=2)

# Or move to a geographic coordinate (degrees; converted kernel-side).
# viewer.goto(lon=-77.0369, lat=38.9072, zoom=4)

## 6. Remove a layer

`viewer.remove_layer(name)` unloads the layer's index and pushes a `REMOVE_LAYER` command to clear it from the viewer.

In [ ]:
viewer.remove_layer("my_detections")
viewer.remove_layer("my_annotations")
print("Layers 'my_detections' and 'my_annotations' removed.")

## Key Takeaways

- **Viewer-first:** open an image, then attach a notebook to the viewer's kernel and `from aws.osml.jupyter import viewer`.
- **Reads are point-in-time**, not reactive — re-run a cell to see the latest `view_bounds` / `last_click`.
- **World enrichment is lazy and memoized:** `.world` triggers one sensor-model call per bounds / click.
- **`.image` vs `.world` vertex order differ:** `.image` is a shapely box (shapely-ordered ring, starts at `(maxx, miny)` CCW); `.world` is explicit TL/TR/BR/BL. Same rectangle, different sequence — use `.image.bounds` for extent, and build image corners yourself to pair them with `.world`.
- **Radians caveat:** `GeodeticWorldCoordinate` is in radians; convert to degrees with `math.degrees()` when calling the sensor model directly. `.world_polygon` already emits degrees.
- **`add_layer` needs a name** and replaces same-named layers — re-running a cell is idempotent.
- **`goto` / `set_view`** move the viewport; geographic coordinates convert kernel-side.